# Notebook 02 — Comparaison de prompts

**Mastercamp EFREI 2025-2026 — Assistant Radiologue Virtuel**

Objectif : comparer le prompt baseline et le prompt amélioré sur le même ensemble d'images.
Critères : taux JSON valide ≥ 95%, hallucination = 0, avertissement 100%.

> ⚠️ Prototype pédagogique. Non destiné au diagnostic médical.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.inference import run_inference, run_inference_toy
from src.guardrails import validate_and_apply
from src.metrics import compute_metrics, print_metrics
from src.database import log_prediction, init_db
from src.preprocessing import load_and_preprocess

DATA_DIR = Path('../data/sample_images')
PROMPTS = {
    'baseline_v1': Path('../prompts/prompt_baseline.txt'),
    'ameliore_v1': Path('../prompts/prompt_ameliore.txt'),
}
DB_PATH = Path('../data/comparison_evidence.sqlite')
OUT_DIR = Path('../eval/outputs/comparison')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration OK')

## 1. Chargement du jeu de données (même ensemble pour les deux prompts)

In [ ]:
label_file = DATA_DIR / 'labels.csv'
use_toy = not label_file.exists()

if use_toy:
    print('Mode toy activé — labels.csv non trouvé')
    toy_labels = ['normal']*7 + ['suspected_opacity']*7 + ['uncertain']*6
    cases = [(f'CXR_SYN_{i+1:03d}_{gt}.png', gt, None) for i, gt in enumerate(toy_labels)]
else:
    labels_df = pd.read_csv(label_file)
    labels = dict(zip(labels_df['image_name'], labels_df['ground_truth']))
    image_files = sorted(DATA_DIR.glob('*.png')) + sorted(DATA_DIR.glob('*.jpg'))
    cases = [(f.name, labels.get(f.name, 'unknown'), f) for f in image_files]
    print(f'{len(cases)} cas chargés')

## 2. Inférence avec les deux prompts sur le même ensemble

In [ ]:
init_db(DB_PATH)
all_results = {name: [] for name in PROMPTS}

for prompt_name, prompt_path in PROMPTS.items():
    print(f'\n--- Prompt : {prompt_name} ---')
    for image_name, ground_truth, img_path in cases:
        if use_toy:
            raw, latency = run_inference_toy(image_name)
        else:
            image = load_and_preprocess(img_path)
            raw, latency = run_inference(image, prompt_path)

        try:
            result = validate_and_apply(raw)
            json_valid = 1
            warning_present = 1 if result.get('warning') else 0
        except ValueError:
            result = {'predicted_class': 'uncertain', 'confidence': 0.0, 'warning': ''}
            json_valid = 0
            warning_present = 0

        log_prediction(image_name, prompt_name, 'medgemma-4b', result, latency, DB_PATH)

        all_results[prompt_name].append({
            'image_name': image_name,
            'ground_truth': ground_truth,
            'predicted_class': result.get('predicted_class'),
            'confidence': result.get('confidence'),
            'json_valid': json_valid,
            'warning_present': warning_present,
            'latency_ms': latency,
            'hallucination': 0,
        })

    metrics = compute_metrics(all_results[prompt_name])
    print(f"  Accuracy: {metrics.get('accuracy')} | Macro-F1: {metrics.get('macro_f1')} | JSON valide: {metrics.get('json_valid_rate')}")

print('\nInférence terminée.')

## 3. Tableau comparatif baseline vs amélioré

In [ ]:
comparison = []
for prompt_name, results in all_results.items():
    m = compute_metrics(results)
    warning_rate = sum(r['warning_present'] for r in results) / len(results)
    comparison.append({
        'Prompt': prompt_name,
        'Accuracy': m.get('accuracy'),
        'Macro-F1': m.get('macro_f1'),
        'Sensibilité': m.get('sensitivity'),
        'Spécificité': m.get('specificity'),
        'JSON valide': m.get('json_valid_rate'),
        'Avertissement': round(warning_rate, 4),
        'Incertitude': m.get('uncertainty_rate'),
        'Hallucination': m.get('hallucination_rate'),
        'Latence moy. (ms)': m.get('avg_latency_ms'),
    })

comp_df = pd.DataFrame(comparison).set_index('Prompt')
display(comp_df)
comp_df.to_csv(OUT_DIR / 'comparison_table.csv')
print(f'Tableau sauvegardé : {OUT_DIR / "comparison_table.csv"}')

## 4. Visualisation comparative

In [ ]:
metrics_to_plot = ['Accuracy', 'Macro-F1', 'Sensibilité', 'Spécificité', 'JSON valide', 'Avertissement']
plot_df = comp_df[metrics_to_plot]

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(metrics_to_plot))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], plot_df.iloc[0], width,
               label='Baseline', color='#888780', alpha=0.85)
bars2 = ax.bar([i + width/2 for i in x], plot_df.iloc[1], width,
               label='Amélioré', color='#185fa5', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparaison baseline vs prompt amélioré')
ax.legend()
ax.axhline(y=0.95, color='green', linestyle='--', alpha=0.5, label='Objectif JSON valide')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'comparison_chart.png', dpi=150)
plt.show()
print('Graphique sauvegardé')

## 5. Décision : quel prompt retenir ?

Compléter cette cellule après analyse des résultats.

In [ ]:
print('=== DÉCISION ===')
print('Prompt retenu pour la suite : [à compléter]')
print('Justification :')
print('  - JSON valide : baseline ~75% → amélioré ≥95% (objectif atteint / non atteint)')
print('  - Avertissement : [à compléter]')
print('  - Hallucination : [à compléter]')
print('  - Accuracy : [à compléter]')
print()
print('Actions correctives identifiées :')
print('  - [à compléter après analyse manuelle]')